# 🚀 AstroLoc-ML · Colab Training (end-to-end)

Idempotent — safe to re-run any cell or **Runtime → Run all** after a session restart.
Every artifact (HYG catalog, checkpoint, figures) is cached to Google Drive so you don't
re-download or re-train after disconnects.

**Setup checklist:**
1. `Runtime → Change runtime type → T4 GPU` (free tier) or **L4 / A100** (Pro).
2. Edit `REPO` in cell 3 to your GitHub repo (just `user/repo`, no URL).
3. Click `Runtime → Run all`. ~20 min on T4.

**Outputs:** trained `checkpoints/best.pt` saved into Google Drive at
`MyDrive/astroloc-ml/checkpoints/`.

## 1 · GPU sanity check

In [ ]:
!nvidia-smi 2>/dev/null | head -16 || echo '❌ No GPU. Runtime > Change runtime type > T4 GPU.'
import torch
print(f'\ntorch={torch.__version__}  cuda_available={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu={torch.cuda.get_device_name(0)}')

## 2 · Mount Google Drive (early, so we can cache everything)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pathlib
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/astroloc-ml')
(DRIVE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'catalogs').mkdir(parents=True, exist_ok=True)
print('Drive cache at:', DRIVE_ROOT)

## 3 · Clone (or update) the repo

`REPO` must be **just `user/repo`** — no `https://github.com/`, no `.git`.
Example: `bachnguyennn/astro-ml`.

In [ ]:
REPO = 'bachnguyennn/astro-ml'   # <-- EDIT ME (just user/repo)

import pathlib, subprocess
%cd /content
repo_dir = pathlib.Path('astroloc-ml')
if repo_dir.exists():
    print('Repo already cloned — pulling latest...')
    !cd astroloc-ml && git pull --ff-only
else:
    !git clone https://github.com/{REPO}.git astroloc-ml
%cd astroloc-ml
!git log --oneline -3
!ls

## 4 · Install Python deps

Colab already has torch + torchvision + numpy + pandas + matplotlib. We only need extras.

In [ ]:
!pip install --quiet astropy seaborn python-dotenv pyyaml piexif

## 5 · HYG star catalog (with Drive cache)

On first run: downloads ~32 MB, then copies to Drive.
Every subsequent run: restores from Drive in <1 s.

In [ ]:
import pathlib, shutil
local_path = pathlib.Path('data/catalogs/hygdata_v3.csv')
drive_path = DRIVE_ROOT / 'catalogs' / 'hygdata_v3.csv'
local_path.parent.mkdir(parents=True, exist_ok=True)

if local_path.exists():
    print(f'✅ Already on disk: {local_path.stat().st_size/1e6:.1f} MB')
elif drive_path.exists():
    shutil.copy(drive_path, local_path)
    print(f'✅ Restored from Drive: {local_path.stat().st_size/1e6:.1f} MB')
else:
    print('Downloading HYG v41 (~32 MB)...')
    !curl -sL -o {local_path} https://raw.githubusercontent.com/astronexus/HYG-Database/main/hyg/CURRENT/hygdata_v41.csv
    shutil.copy(local_path, drive_path)
    print(f'✅ Downloaded + cached: {local_path.stat().st_size/1e6:.1f} MB')

!head -1 {local_path} | cut -c1-120; echo

## 6 · Verify the pipeline is wired up correctly

Loads the catalog, renders one sample, and runs a forward pass.
Should take ~5 s. If it fails here, training would just waste minutes — fix the error first.

In [ ]:
import sys, time, torch, numpy as np
sys.path.insert(0, '/content/astroloc-ml')
from src.data.catalog import load_hyg_catalog
from src.data.renderer import render_star_field
from src.models.astrolocnet import AstroLocNet

cat = load_hyg_catalog('data/catalogs/hygdata_v3.csv', mag_limit=8.0)
print(f'Loaded {len(cat):,} stars')

t = time.perf_counter()
img = render_star_field(85.0, -2.0, 30.0, 0.0, cat, image_size=224, rng=np.random.default_rng(0))
print(f'Rendered one star field in {(time.perf_counter()-t)*1000:.0f} ms, shape={img.shape}')

model = AstroLocNet(pretrained=False).cuda()
x = torch.randn(2, 3, 224, 224, device='cuda')
y = model(x)
print(f'Forward pass OK: in={tuple(x.shape)} out={tuple(y.shape)} (B, [ra, dec, rot, log_scale])')

## 7 · Confirm we have the new CLI flags

If you don't see `--train-samples` and `--num-workers` below, your repo is missing the
latest commit — push the local changes from your laptop and re-run cell 3.

In [ ]:
!python train.py --help 2>&1 | grep -E 'train-samples|num-workers|epochs-phase' || \
  echo '❌ Missing flags. Run `git push` on your laptop, then re-run cell 3.'

## 8 · Smoke training (~30 s on T4)

Verifies the full train loop runs end-to-end with tiny data. Skip if you've done it before.

In [ ]:
!python train.py --config configs/default.yaml --smoke --skip-phase3 --device cuda

## 9 · Real training run

Training is **CPU-bound** (synthetic renderer runs in DataLoader workers on Colab's ~2 vCPUs),
not GPU-bound. Wall-clock scales linearly with `--train-samples`.

| Preset            | `--train-samples` | Epochs (p1/p2) | T4 wall-clock |
| ----------------- | ----------------- | -------------- | ------------- |
| **Fast (active)** | 5,000             | 3 / 6          | **~15–25 min**|
| Standard          | 20,000            | 5 / 10         | ~1.5–2 h      |
| Full (default)    | 50,000            | 5 / 15         | ~5–7 h        |

Output is streamed live — you'll see JSON lines like `{"phase":"phase1","epoch":1,...}` every
couple minutes. **Do not pipe through `tail`** — it buffers stdout and hides progress.

In [ ]:
# === FAST preset (recommended for first run) — ~15-25 min on T4 ===
!python -u train.py --config configs/default.yaml --device cuda --skip-phase3 \
  --train-samples 5000 --val-samples 500 --num-workers 2 \
  --epochs-phase1 3 --epochs-phase2 6

# === STANDARD preset (~1.5-2 h) — uncomment to use instead ===
# !python -u train.py --config configs/default.yaml --device cuda --skip-phase3 \
#   --train-samples 20000 --val-samples 1000 --num-workers 2 \
#   --epochs-phase1 5 --epochs-phase2 10

# === FULL preset (~5-7 h) — only if you really want to push it ===
# !python -u train.py --config configs/default.yaml --device cuda --skip-phase3 --num-workers 2

## 10 · Inspect the trained checkpoint

In [ ]:
import torch, json
ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
print('Best val angular separation (deg):', round(ckpt['best_metric'], 3))
print('History epochs:', len(ckpt.get('history', [])))
if ckpt.get('history'):
    last = ckpt['history'][-1]
    print('Last entry:', json.dumps({'phase': last['phase'], 'epoch': last['epoch'],
                                     'val': last['val']}, indent=2, default=str))

## 11 · Save checkpoint + figures to Google Drive

Survives session disconnects. Re-running cell 11 always overwrites the newest file.

In [ ]:
import shutil, time, pathlib
out = DRIVE_ROOT / 'checkpoints'
out.mkdir(parents=True, exist_ok=True)
stamp = time.strftime('%Y%m%d_%H%M%S')

shutil.copy('checkpoints/best.pt', out / f'best_{stamp}.pt')
shutil.copy('checkpoints/best.pt', out / 'best_latest.pt')
print(f'✅ Saved checkpoint: {out / f"best_{stamp}.pt"}')
print(f'✅ Saved alias:      {out / "best_latest.pt"}')

## 12 · Regenerate README figures from the trained model

In [ ]:
!python scripts/generate_readme_figures.py
!ls -lh reports/figures/

In [ ]:
from IPython.display import Image, display
for p in ['reports/figures/05_training_curves.png',
          'reports/figures/09_demo_overlay_0.png',
          'reports/figures/09_demo_overlay_1.png',
          'reports/figures/09_demo_overlay_2.png']:
    display(Image(p))

In [ ]:
# Copy fresh figures to Drive too
import shutil, pathlib
fig_out = DRIVE_ROOT / 'figures'
fig_out.mkdir(parents=True, exist_ok=True)
for src in pathlib.Path('reports/figures').glob('*.png'):
    shutil.copy(src, fig_out / src.name)
print('✅ Figures copied to', fig_out)

## 13 · Optional: download checkpoint to your laptop

In [ ]:
from google.colab import files
files.download('checkpoints/best.pt')

---
## Reconnecting after a disconnect

If Colab killed your session, just **Runtime → Run all** again.
Cell 2 re-mounts Drive, cell 5 restores HYG from Drive cache (<1 s), and cell 11 keeps your
previous checkpoint safe in Drive at `MyDrive/astroloc-ml/checkpoints/best_latest.pt`.

To skip retraining and reuse a saved checkpoint:

```python
import shutil
shutil.copy(DRIVE_ROOT / 'checkpoints/best_latest.pt', 'checkpoints/best.pt')
```

Then jump straight to cell 10 (inspect) and cell 12 (figures).